# Scrape, Translate, and Summarize a News Article Using Frontier LLM Models
This notebook demonstrates how to scrape a webpage, translate the article into English, and then summarize the translated content using Groq with a fallback to OpenAI.

In [ ]:

from pathlib import Path
import importlib
import os
import sys

from dotenv import load_dotenv
from groq import Groq
from openai import OpenAI

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

import src.services.scraper as scraper_module
scraper_module = importlib.reload(scraper_module)
from src.services.scraper import scrape_article

# Section 1: Scrape the article from the provided URL.
url = "https://www.bbc.com/zhongwen/articles/cjwgxdz5n49o/trad"

try:
    article = scrape_article(url)
    print(article[:4000])
except Exception as exc:
    print(f"Scraping failed: {exc}")

中國能複製電動車成功經驗到「機器人的士」嗎?
- Author, 蘇蘭嘉娜·特瓦里（Suranjana Tewari）
- Role, BBC 亞洲商務記者
- Reporting from, 北京報導
 
- Published
- 閱讀時間: 4 分鐘
在中國北京亦莊區，無人駕駛車輛已成為常見景象。機器人的士（robotaxi；機器人計程車／機器人出租車）穿梭在車流中，與普通汽車並行，而自動駕駛送貨車則在內側車道平穩行駛，將包裹運送到取件點。
這個區域已成為中國自動駕駛技術的重要測試場之一，百度、文遠知行（WeRide）和小馬智行（Pony.ai）等公司在指定區域內提供商業化的機器人的士服務。
叫車只需打開手機應用程式，短短幾分鐘內，一輛無人的士就會抵達，駕駛座上空無一人。在觸控螢幕上確認行程後，車輛便融入北京繁忙的車流中，穿梭於公車、腳踏車、機車和行人之間，幾乎毫不遲疑。
這項技術仍在持續演進。但，更大的問題已浮現：中國企業能否像主導電動車（EV）市場一樣，讓機器人的士成為另一個全球領先的領域？
自動駕駛車崛起中國市場
中國的自動駕駛企業已擁有強大的優勢，那就是幫助中國成為全球最大電動車市場的產業生態系統。
不過，與大部分技術都由自家設計的特斯拉不同，中國的自駕產業是建立在企業網絡基礎上的。
譬如，比亞迪、奇瑞、吉利和上汽等老牌車廠負責製造車輛，而專業公司則專注開發軟體。
事實上，自動駕駛車輛仰賴與電動車相同的電池、感測器、晶片和車載電腦等關鍵部件。而且，由於這些供應鏈已存在龐大規模，企業得以更快、更低成本地開發技術。
「你看到的是中國電動車產業的創新與適應速度，我認為世界其他地方都難以匹敵。」布魯金斯學會（Brookings Institution）外交政策研究員陳凱欣（Kyle Chan）告訴BBC。
陳凱欣強調，「中國的電動車產能並非僅止於此，它還透過我稱之的『重疊技術產業生態系統』（overlapping tech industrial ecosystems），溢出到其他相關產業。」
新生產力
中國政府政策也扮演重要角色。多個城市的試點計畫允許企業在部分公共道路上測試這項技術。
但是，中國還為致力讓技術更聰明的企業提供了另一項優勢：複雜的駕駛環境。
在北京開一段路，自動駕駛車輛可能就需要應對公車、機車、腳踏車、行人以及難以預測的車流。
「中國這邊的交通

#### Translate the scraped article into English using ***llama-3.3-70b-versatile***

In [25]:

if "article" in locals() and article:
    groq_api_key = os.getenv("GROQ_API_KEY")
    openai_api_key = os.getenv("OPENAI_API_KEY")

    def call_llm(prompt: str, preferred_provider: str = "groq") -> str:
        providers = []
        if preferred_provider == "groq" and groq_api_key:
            providers.append(("groq", groq_api_key))
        if preferred_provider == "openai" and openai_api_key:
            providers.append(("openai", openai_api_key))

        if preferred_provider == "groq" and openai_api_key:
            providers.append(("openai", openai_api_key))
        if preferred_provider == "openai" and groq_api_key:
            providers.append(("groq", groq_api_key))

        for provider_name, api_key in providers:
            try:
                if provider_name == "groq":
                    client = Groq(api_key=api_key)
                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[
                            {"role": "system", "content": "You are a precise translator and summarizer."},
                            {"role": "user", "content": prompt},
                        ],
                        temperature=0.2,
                    )
                    return response.choices[0].message.content.strip()

                if provider_name == "openai":
                    client = OpenAI(api_key=api_key)
                    response = client.responses.create(
                        model="gpt-4o-mini",
                        instructions="You are a precise translator and summarizer.",
                        input=prompt,
                    )
                    return response.output_text.strip()
            except Exception as exc:
                print(f"{provider_name.title()} failed: {exc}")

        raise RuntimeError("No working LLM provider available")

    text = article.strip()
    if len(text) > 12000:
        text = text[:12000]

    translation_prompt = (
        "Translate the following article to accurate, natural English. "
        "Preserve the meaning, names, numbers, dates, and key facts.\n\n"
        f"{text}"
    )
    translation = call_llm(translation_prompt, preferred_provider="groq")

    print("\nTranslation:\n")
    print(translation)
else:
    print("No article text was available for translation.")


Translation:

Can China Replicate its Electric Vehicle Success with "Robotaxis"?

By Suranjana Tewari, BBC Asia Business Correspondent, reporting from Beijing

In Beijing's Yizhuang district, driverless vehicles have become a common sight. Robotaxis, also known as self-driving taxis, navigate through traffic alongside regular cars, while autonomous delivery vans drive smoothly on the inner lanes, delivering packages to pickup points.

This area has become one of China's key testing grounds for autonomous driving technology, with companies like Baidu, WeRide, and Pony.ai offering commercial robotaxi services within designated areas.

Hailing a ride is as simple as opening a mobile app, and within minutes, a driverless taxi arrives, with no one in the driver's seat. After confirming the trip on a touchscreen, the vehicle merges into Beijing's busy traffic, weaving through buses, bicycles, motorcycles, and pedestrians with ease.

While the technology is still evolving, a bigger question 

#### Summarize the translated article in English using ***llama-3.3-70b-versatile***

In [26]:

if "translation" in locals() and translation:
    groq_api_key = os.getenv("GROQ_API_KEY")
    openai_api_key = os.getenv("OPENAI_API_KEY")

    def call_llm(prompt: str, preferred_provider: str = "groq") -> str:
        providers = []
        if preferred_provider == "groq" and groq_api_key:
            providers.append(("groq", groq_api_key))
        if preferred_provider == "openai" and openai_api_key:
            providers.append(("openai", openai_api_key))

        if preferred_provider == "groq" and openai_api_key:
            providers.append(("openai", openai_api_key))
        if preferred_provider == "openai" and groq_api_key:
            providers.append(("groq", groq_api_key))

        for provider_name, api_key in providers:
            try:
                if provider_name == "groq":
                    client = Groq(api_key=api_key)
                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[
                            {"role": "system", "content": "You are a concise summarizer."},
                            {"role": "user", "content": prompt},
                        ],
                        temperature=0.2,
                    )
                    return response.choices[0].message.content.strip()

                if provider_name == "openai":
                    client = OpenAI(api_key=api_key)
                    response = client.responses.create(
                        model="gpt-4o-mini",
                        instructions="You are a concise summarizer.",
                        input=prompt,
                    )
                    return response.output_text.strip()
            except Exception as exc:
                print(f"{provider_name.title()} failed: {exc}")

        raise RuntimeError("No working LLM provider available")

    summary_prompt = (
        "Based on the translated article below, write a concise summary in English with:\n"
        "- 3 bullet points of the main facts\n"
        "- 1 short concluding sentence\n\n"
        "Translated article:\n"
        f"{translation}"
    )
    summary = call_llm(summary_prompt, preferred_provider="groq")

    print("\nSummary:\n")
    print(summary)
else:
    print("No translated article was available for summarization.")


Summary:

Here is a concise summary of the article in 3 bullet points and a concluding sentence:

* China's autonomous driving industry has a strong advantage due to its existing ecosystem, which helped the country become the world's largest electric vehicle market, with companies like Baidu and WeRide offering commercial robotaxi services.
* The Chinese government has played a crucial role in supporting the industry through pilot programs and regulations, and the country's complex driving environment provides a unique advantage for testing and improving autonomous vehicles.
* Despite challenges such as safety concerns and regulatory barriers, Chinese companies are rapidly expanding globally, partnering with international companies and competing with US-based companies like Waymo and Tesla.

China is poised to replicate its electric vehicle success with "robotaxis" and become a global leader in autonomous driving technology.
